# Instance Detection Head

Training-free instance detection via DINOv3 patch-token cosine similarity.
Given one **exemplar** image (what the object looks like) and a **query** image
(the scene to search), the pipeline:

1. Extracts L2-normalised patch tokens from an intermediate transformer block.
2. **Masks** the exemplar tokens to the object region using the accompanying `_mask_good.npz`.
3. Aggregates the *masked* exemplar tokens into a compact descriptor (mean **or** K-means centroids).
4. Computes per-patch cosine similarity between query and descriptor → density map.
5. Applies background suppression (threshold) and max-pool NMS to find instance centres.

| Step | Function |
|------|----------|
| Token extraction | `extract_patch_tokens` |
| **Mask → patch grid** | `pixel_mask_to_patch_mask` (notebook helper) |
| Exemplar aggregation | `compute_exemplar_features` |
| Density map | `compute_density_map` |
| Peak extraction (NMS) | `extract_peaks` |

In [ ]:
# Logging must be configured before torch is imported.
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s: %(message)s",
    force=True,
)
log = logging.getLogger("instance_detection")

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.decomposition import PCA
from dotenv import load_dotenv

from dinoisawesome import DinoEncoder
from dinoisawesome.instance_detection import (
    compute_density_map,
    compute_exemplar_features,
    extract_patch_tokens,
    extract_peaks,
)
from pathlib import Path
from glob import glob

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
load_dotenv()

# Exemplar image + its instance mask (.npy, shape (N, H, W) bool).
# All instance channels are unioned → single foreground mask.
part_type = "LHb"
ref_number = 1
data_dir = Path(f"DinoisAwesome/data/abc2")

# Reference image — set to a local file path string.
all_images = glob(str(data_dir / f"{part_type}_*.jpg"))
REF_IMAGE_PATH: str = [i for i in all_images if f"{part_type}_{str(ref_number)}.jpg" in i][0]
TAR_IMAGE_PATH: list[str] = [i for i in all_images if f"{part_type}_{str(ref_number)}.jpg" not in i]

# Resolve exemplar / query / mask paths from the parameters above.
EXEMPLAR_PATH: str = REF_IMAGE_PATH
QUERY_PATHS: list[str] = TAR_IMAGE_PATH
EXEMPLAR_MASK_PATH: str | None = str(data_dir / "annotations" / f"{part_type}_{ref_number}.npy")

# Model settings
DINO_VERSION = "v3"  # "v2" or "v3"
DINO_SIZE = "large"  # "small" | "base" | "large" | "giant"
IMG_SIZE = 1024  # must be divisible by patch_size (16 for v3, 14 for v2)
LAYER_IDX = 23  # 0-based transformer block index

# Mask → patch-grid: a patch is "in mask" if this fraction of its pixels are True.
MASK_PATCH_THRESHOLD = 0.3

# Exemplar aggregation
EXEMPLAR_MODE = "mean"  # "mean" or "kmeans"
EXEMPLAR_K = 3  # number of centroids (only used when mode="kmeans")

# Density map / NMS
DENSITY_THRESHOLD = 0.3
PEAK_KERNEL_SIZE = 5
MIN_PEAK_THRESHOLD = 0.01

# Local weights directory
DINO_WEIGHTS_DIR: str | None = os.environ.get("DINO_WEIGHTS_DIR")
print(f"number of reference images: 1")
print(f"number of target images: {len(TAR_IMAGE_PATH)}")

In [ ]:
# ── Mask helpers ──────────────────────────────────────────────────────────────


def load_pixel_mask(npz_path: str) -> np.ndarray:
    """Load a _mask_good.npz, union all instance channels → (H, W) bool."""
    seg = np.load(npz_path)["segmaps"]  # (H, W, N)
    return seg.any(axis=2)  # (H, W)


def pixel_mask_to_patch_mask(
    pixel_mask: np.ndarray,
    grid_h: int,
    grid_w: int,
    img_size: int,
    threshold: float = 0.3,
) -> np.ndarray:
    """Resize a pixel-space boolean mask to patch-grid resolution.

    Steps
    -----
    1. Resize the pixel mask to ``img_size × img_size`` (nearest-neighbour).
    2. Partition into ``grid_h × grid_w`` patch windows.
    3. A patch is True if >= ``threshold`` fraction of its pixels are True.

    Returns
    -------
    (grid_h, grid_w) bool ndarray
    """
    # Step 1 — resize to encoder input resolution
    mask_pil = Image.fromarray(pixel_mask.astype(np.uint8) * 255)
    mask_resized = (
        np.array(mask_pil.resize((img_size, img_size), Image.NEAREST)) > 0
    )  # (img_size, img_size) bool

    # Step 2 — pool to patch grid
    ph = img_size // grid_h  # patch height in pixels
    pw = img_size // grid_w
    # Reshape to (grid_h, ph, grid_w, pw) then average over the two patch axes
    tiled = mask_resized.reshape(grid_h, ph, grid_w, pw)
    patch_density = tiled.mean(axis=(1, 3))  # (grid_h, grid_w) float
    return patch_density >= threshold  # (grid_h, grid_w) bool

In [ ]:
# ── Shared visualisation helpers ───────────────────────────────────────────────


def upsample_map(arr: np.ndarray, size: int) -> np.ndarray:
    """Nearest-neighbour upsample a (H,W) float map, normalised to [0,1]."""
    norm = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    pil = Image.fromarray((norm * 255).astype(np.uint8))
    return np.array(pil.resize((size, size), Image.NEAREST)) / 255.0


def heat_overlay(bg: np.ndarray, heat: np.ndarray, alpha: float = 0.55) -> np.ndarray:
    """Blend a [0,1] normalised jet heatmap over an uint8 RGB image."""
    colored = plt.get_cmap("jet")(heat)[..., :3]
    return np.clip(bg / 255.0 * (1 - alpha) + colored * alpha, 0, 1)

In [ ]:
# ── Load encoder ───────────────────────────────────────────────────────────────
encoder = DinoEncoder(
    version=DINO_VERSION,
    size=DINO_SIZE,
    img_size=IMG_SIZE,
    weights_dir=DINO_WEIGHTS_DIR,
)
log.info(
    "DINOv%s-%s | patch_size=%d | grid=%dx%d",
    DINO_VERSION[1],
    DINO_SIZE,
    encoder.patch_size,
    encoder.grid_h,
    encoder.grid_w,
)

GRID_H, GRID_W = encoder.grid_h, encoder.grid_w
PATCH_SIZE = encoder.patch_size

## Step 0 — Input Images and Exemplar Mask

Show the exemplar with its instance mask overlaid, alongside the query images.
The mask selects which part of the exemplar we actually want to match — only those
patch tokens feed into the descriptor.

In [ ]:
# ── Load images ────────────────────────────────────────────────────────────────
exemplar_img = Image.open(EXEMPLAR_PATH).convert("RGB")
query_imgs = [Image.open(p).convert("RGB") for p in QUERY_PATHS]

log.info("Exemplar: %s  (%dx%d px)", EXEMPLAR_PATH, *exemplar_img.size)
for p, qi in zip(QUERY_PATHS, query_imgs):
    log.info("Query:    %s  (%dx%d px)", p, *qi.size)

# ── Load and inspect pixel mask ────────────────────────────────────────────────
pixel_mask: np.ndarray | None = None
if EXEMPLAR_MASK_PATH is not None:
    if EXEMPLAR_MASK_PATH.endswith(".npy"):
        raw_seg = np.load(EXEMPLAR_MASK_PATH)  # (N, H, W) bool
        raw_seg = raw_seg.transpose(1, 2, 0)  # → (H, W, N)
    elif EXEMPLAR_MASK_PATH.endswith(".npz"):
        raw_seg = np.load(EXEMPLAR_MASK_PATH)["segmaps"]  # (H, W, N)
    else:
        raw_seg = Image.open(EXEMPLAR_MASK_PATH).convert("L")  # grayscale
        raw_seg = np.array(raw_seg)[:, :, None] > 0
    pixel_mask = raw_seg.any(axis=2)  # (H, W) union
    n_instances = raw_seg.shape[2]
    log.info(
        "Mask: %s  shape=%s  instances=%d  coverage=%.1f%%",
        EXEMPLAR_MASK_PATH,
        raw_seg.shape,
        n_instances,
        100.0 * pixel_mask.mean(),
    )

# ── Visualise exemplar + mask + queries ───────────────────────────────────────
display_ex = np.array(exemplar_img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

ncols = 2 + len(query_imgs)  # raw exemplar | masked exemplar | queries
fig, axes = plt.subplots(1, ncols, figsize=(ncols * 4, 5))

# Panel 0: raw exemplar
axes[0].imshow(exemplar_img)
axes[0].set_title("Exemplar (original)", fontsize=10)
axes[0].axis("off")

# Panel 1: exemplar with mask overlay
axes[1].imshow(display_ex)
if pixel_mask is not None:
    # Resize mask to IMG_SIZE for display
    mask_disp = (
        np.array(
            Image.fromarray(pixel_mask.astype(np.uint8) * 255).resize(
                (IMG_SIZE, IMG_SIZE), Image.NEAREST
            )
        )
        > 0
    )
    # Green overlay inside mask
    green_overlay = np.zeros((*mask_disp.shape, 4), dtype=np.float32)
    green_overlay[mask_disp] = [0.2, 0.9, 0.2, 0.5]
    axes[1].imshow(green_overlay)
    axes[1].set_title(
        f"Exemplar + instance mask\n"
        f"{n_instances} instance(s) | {100 * pixel_mask.mean():.1f}% coverage",
        fontsize=10,
    )
else:
    axes[1].set_title("Exemplar (no mask)", fontsize=10)
axes[1].axis("off")

# Remaining panels: queries
for i, (p, qi) in enumerate(zip(QUERY_PATHS, query_imgs), start=2):
    axes[i].imshow(qi)
    axes[i].set_title(f"Query {i - 1}\n{p.split('/')[-1]}", fontsize=9)
    axes[i].axis("off")

plt.suptitle("Step 0: Input images and exemplar mask", fontsize=12)
plt.tight_layout()
plt.show()

## Step 1 — Patch Token Extraction + Mask Projection

`extract_patch_tokens` gives us `(H×W, D)` L2-normalised tokens for each image.

The pixel-space mask (768×768) is then:
1. **Resized** to `IMG_SIZE × IMG_SIZE` (nearest-neighbour).
2. **Pooled** to the patch grid: a patch is *inside* the mask if ≥ `MASK_PATCH_THRESHOLD`
   of its pixels are True.

Only the **masked** tokens feed into the exemplar descriptor — background patches are discarded.

We project tokens into 3D via PCA (fitted jointly on exemplar + query) and colour them
to show the feature landscape; masked patches are highlighted with a border overlay.

In [ ]:
# ── Extract tokens ─────────────────────────────────────────────────────────────
exemplar_tokens, ex_h, ex_w = extract_patch_tokens(encoder, exemplar_img, LAYER_IDX)
query_tokens, q_h, q_w = extract_patch_tokens(encoder, query_imgs[0], LAYER_IDX)

log.info(
    "Exemplar tokens: %s  (grid %dx%d)  device=%s",
    exemplar_tokens.shape,
    ex_h,
    ex_w,
    exemplar_tokens.device,
)
log.info("Query tokens:    %s  (grid %dx%d)", query_tokens.shape, q_h, q_w)

# ── Build patch-grid mask ──────────────────────────────────────────────────────
if pixel_mask is not None:
    patch_mask = pixel_mask_to_patch_mask(
        pixel_mask, ex_h, ex_w, IMG_SIZE, threshold=MASK_PATCH_THRESHOLD
    )  # (ex_h, ex_w) bool
    patch_mask_flat = torch.from_numpy(patch_mask.reshape(-1)).to(exemplar_tokens.device)  # (N,)
    exemplar_tokens_masked = exemplar_tokens[patch_mask_flat]  # (M, D)
    log.info(
        "Patch mask: %d / %d patches inside mask  (threshold=%.2f)",
        patch_mask_flat.sum().item(),
        ex_h * ex_w,
        MASK_PATCH_THRESHOLD,
    )
else:
    patch_mask = np.ones((ex_h, ex_w), dtype=bool)
    patch_mask_flat = torch.ones(ex_h * ex_w, dtype=torch.bool, device=exemplar_tokens.device)
    exemplar_tokens_masked = exemplar_tokens
    log.info("No mask — using all %d exemplar patches.", ex_h * ex_w)

In [ ]:
# ── PCA visualisation: joint embedding ────────────────────────────────────────
all_tokens_np = torch.cat([exemplar_tokens, query_tokens], dim=0).cpu().float().numpy()
pca = PCA(n_components=3)
pca.fit(all_tokens_np)
log.info(
    "PCA explained variance: %.1f%% / %.1f%% / %.1f%%",
    pca.explained_variance_ratio_[0] * 100,
    pca.explained_variance_ratio_[1] * 100,
    pca.explained_variance_ratio_[2] * 100,
)


def tokens_to_pca_rgb(tokens: torch.Tensor, h: int, w: int) -> np.ndarray:
    proj = pca.transform(tokens.cpu().float().numpy())  # (N, 3)
    for c in range(3):
        lo, hi = proj[:, c].min(), proj[:, c].max()
        proj[:, c] = (proj[:, c] - lo) / (hi - lo + 1e-8)
    return proj.reshape(h, w, 3)


ex_rgb = tokens_to_pca_rgb(exemplar_tokens, ex_h, ex_w)  # (H, W, 3)
q_rgb = tokens_to_pca_rgb(query_tokens, q_h, q_w)


# Upscale PCA maps to IMG_SIZE (nearest so patch boundaries stay sharp)
def to_display(rgb_hw3: np.ndarray, size: int) -> np.ndarray:
    return (
        np.array(
            Image.fromarray((rgb_hw3 * 255).astype(np.uint8)).resize((size, size), Image.NEAREST)
        )
        / 255.0
    )


ex_rgb_up = to_display(ex_rgb, IMG_SIZE)
q_rgb_up = to_display(q_rgb, IMG_SIZE)

display_q0 = np.array(query_imgs[0].resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

# Patch-grid mask upscaled for overlay
mask_border = (
    np.array(
        Image.fromarray(patch_mask.astype(np.uint8) * 255).resize(
            (IMG_SIZE, IMG_SIZE), Image.NEAREST
        )
    )
    > 0
)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 0 — exemplar
axes[0, 0].imshow(display_ex)
axes[0, 0].set_title("Exemplar — original", fontsize=10)
axes[0, 0].axis("off")

axes[0, 1].imshow(ex_rgb_up)
axes[0, 1].set_title(
    f"Exemplar — PCA(tokens) → RGB\nblock {LAYER_IDX}  |  grid {ex_h}×{ex_w}",
    fontsize=10,
)
axes[0, 1].axis("off")

# PCA map with mask region highlighted
axes[0, 2].imshow(ex_rgb_up)
green_ov = np.zeros((IMG_SIZE, IMG_SIZE, 4), dtype=np.float32)
green_ov[mask_border] = [0.2, 0.9, 0.2, 0.45]
green_ov[~mask_border] = [0.0, 0.0, 0.0, 0.35]  # darken excluded patches
axes[0, 2].imshow(green_ov)
axes[0, 2].set_title(
    f"Exemplar PCA — masked patches highlighted\n"
    f"{patch_mask.sum()} / {ex_h * ex_w} patches used  (threshold={MASK_PATCH_THRESHOLD})",
    fontsize=10,
)
axes[0, 2].axis("off")

# Row 1 — query (reference, no mask)
axes[1, 0].imshow(display_q0)
axes[1, 0].set_title("Query 1 — original", fontsize=10)
axes[1, 0].axis("off")

axes[1, 1].imshow(q_rgb_up)
axes[1, 1].set_title(
    f"Query 1 — PCA(tokens) → RGB\nblock {LAYER_IDX}  |  grid {q_h}×{q_w}",
    fontsize=10,
)
axes[1, 1].axis("off")
axes[1, 2].axis("off")


plt.suptitle(
    f"Step 1: Patch token extraction + mask projection  |  DINOv{DINO_VERSION[1]}-{DINO_SIZE}",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Step 2 — Exemplar Feature Aggregation (masked tokens only)

`compute_exemplar_features` now receives only the **M masked tokens** instead of all H×W.

The visualisation compares:
- **Mean descriptor** — single centroid of the masked tokens.
- **K-means** — K centroids; each centroid captures a distinct part of the object.

The similarity map shows how similar each *query* patch is to each centroid —
letting you see which parts of the descriptor fire where in the scene.

In [ ]:
# ── Mean descriptor (masked) ──────────────────────────────────────────────────
feat_mean = compute_exemplar_features(exemplar_tokens_masked, mode="mean")
log.info("Mean descriptor shape: %s", feat_mean.shape)

# Per-exemplar-patch similarity to the mean — visualise how compact the masked region is
sim_ex_to_mean = (exemplar_tokens @ feat_mean.T).squeeze(1).reshape(ex_h, ex_w).cpu().numpy()

# ── K-means descriptors (masked) ──────────────────────────────────────────────
feat_kmeans = compute_exemplar_features(exemplar_tokens_masked, mode="kmeans", k=EXEMPLAR_K)
log.info("K-means descriptors shape: %s  (K=%d)", feat_kmeans.shape, EXEMPLAR_K)

# Centroid assignment map across ALL exemplar patches (masked + unmasked, for visualisation)
sim_ex_km = (exemplar_tokens @ feat_kmeans.T).cpu().numpy()  # (N, K)
assignments = sim_ex_km.argmax(axis=1).reshape(ex_h, ex_w)
max_sim_km = sim_ex_km.max(axis=1).reshape(ex_h, ex_w)

# ── Visualise ─────────────────────────────────────────────────────────────────
CMAP_KM = plt.get_cmap("tab10")

fig, axes = plt.subplots(1, 4, figsize=(25, 5))

# 0: exemplar with mask
axes[0].imshow(display_ex)
axes[0].imshow(green_ov)
axes[0].set_title("Exemplar + patch mask", fontsize=10)
axes[0].axis("off")

# 1: similarity to mean (on exemplar)
im1 = axes[1].imshow(
    sim_ex_to_mean, cmap="viridis", vmin=sim_ex_to_mean.min(), vmax=1.0, aspect="auto"
)
axes[1].set_title("Exemplar patches: sim to mean descriptor\n(masked-token mean)", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], shrink=0.75, pad=0.02)

# 2: K-means centroid assignment
axes[2].imshow(assignments, cmap="tab10", vmin=0, vmax=9, aspect="auto")
axes[2].imshow(  # darken excluded
    np.where(patch_mask[:, :, None], 0, 0.4).repeat(4, axis=2).astype(np.float32)
    * np.array([0, 0, 0, 1]),
)
legend_km = [mpatches.Patch(color=CMAP_KM(i), label=f"centroid {i}") for i in range(EXEMPLAR_K)]
axes[2].legend(handles=legend_km, loc="lower right", fontsize=8, framealpha=0.85)
axes[2].set_title(
    f"K-means centroid assignment (K={EXEMPLAR_K})\nmasked-token K-means", fontsize=10
)
axes[2].axis("off")

# 3: max similarity to nearest centroid
im3 = axes[3].imshow(max_sim_km, cmap="viridis", vmin=max_sim_km.min(), vmax=1.0, aspect="auto")
axes[3].set_title(f"Max sim to nearest centroid (K={EXEMPLAR_K})", fontsize=10)
axes[3].axis("off")
plt.colorbar(im3, ax=axes[3], shrink=0.75, pad=0.02)

plt.suptitle("Step 2: Exemplar feature aggregation (masked tokens)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Step 3 — Cosine Similarity Density Map

`compute_density_map` compares every query patch to the (masked) exemplar descriptor(s).

We also show a **masked vs unmasked** comparison: the same pipeline run without the mask
so you can see how background tokens in the descriptor pollute the density map.

In [ ]:
# ── Compute density maps: masked vs unmasked ──────────────────────────────────
feat_masked = compute_exemplar_features(exemplar_tokens_masked, mode=EXEMPLAR_MODE, k=EXEMPLAR_K)
feat_unmasked = compute_exemplar_features(exemplar_tokens, mode=EXEMPLAR_MODE, k=EXEMPLAR_K)


# Raw similarity (before threshold) for comparison
def raw_sim_2d(q_tok, feat):
    return (q_tok @ feat.T).mean(dim=-1).reshape(q_h, q_w).cpu().numpy()


sim_masked_raw = raw_sim_2d(query_tokens, feat_masked)
sim_unmasked_raw = raw_sim_2d(query_tokens, feat_unmasked)

dm_masked = compute_density_map(query_tokens, feat_masked, q_h, q_w, DENSITY_THRESHOLD)
dm_unmasked = compute_density_map(query_tokens, feat_unmasked, q_h, q_w, DENSITY_THRESHOLD)

log.info(
    "Masked   density: max=%.3f  non-zero=%d/%d patches",
    dm_masked.max().item(),
    (dm_masked > 0).sum().item(),
    q_h * q_w,
)
log.info(
    "Unmasked density: max=%.3f  non-zero=%d/%d patches",
    dm_unmasked.max().item(),
    (dm_unmasked > 0).sum().item(),
    q_h * q_w,
)

# ── Figure: 2 rows (masked | unmasked) × 3 cols (raw sim | density | overlay) ─
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row, (label, sim_raw, dm, feat) in enumerate(
    [
        ("masked exemplar", sim_masked_raw, dm_masked, feat_masked),
        ("unmasked exemplar", sim_unmasked_raw, dm_unmasked, feat_unmasked),
    ]
):
    dm_np = dm.cpu().numpy()

    im0 = axes[row, 0].imshow(sim_raw, cmap="jet", aspect="auto")
    axes[row, 0].set_title(f"[{label}]\nRaw cosine similarity", fontsize=10)
    axes[row, 0].axis("off")
    plt.colorbar(im0, ax=axes[row, 0], shrink=0.75, pad=0.02)

    im1 = axes[row, 1].imshow(dm_np, cmap="jet", aspect="auto")
    axes[row, 1].set_title(f"[{label}]\nDensity map (threshold={DENSITY_THRESHOLD})", fontsize=10)
    axes[row, 1].axis("off")
    plt.colorbar(im1, ax=axes[row, 1], shrink=0.75, pad=0.02)

    axes[row, 2].imshow(display_q0)
    axes[row, 2].imshow(heat_overlay(display_q0, upsample_map(dm_np, IMG_SIZE)))
    axes[row, 2].set_title(f"[{label}]\nDensity overlay on query", fontsize=10)
    axes[row, 2].axis("off")

plt.suptitle(
    f"Step 3: Density map — masked vs unmasked exemplar  |  mode={EXEMPLAR_MODE}",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Step 4 — Max-Pool NMS Peak Extraction

`extract_peaks` locates local maxima via max-pool NMS:

1. Run `max_pool2d` with a `kernel_size × kernel_size` window.
2. A patch is a **peak** if its density equals the pooled value (local max)
   AND density > `min_peak_threshold`.

We expose all intermediate tensors so you can see exactly which patches survive.

In [ ]:
MIN_PEAK_THRESHOLD = 0.3

In [ ]:
# ── Reproduce NMS internals ────────────────────────────────────────────────────
density_map = dm_masked  # use the masked-exemplar density map from here on

dm_4d = density_map.unsqueeze(0).unsqueeze(0)
padding = PEAK_KERNEL_SIZE // 2
pooled = F.max_pool2d(dm_4d, kernel_size=PEAK_KERNEL_SIZE, stride=1, padding=padding).squeeze()

is_local_max = density_map == pooled
is_peak = is_local_max & (density_map > MIN_PEAK_THRESHOLD)

peaks = extract_peaks(density_map, PEAK_KERNEL_SIZE, MIN_PEAK_THRESHOLD)
peaks_sim = density_map[peaks[:, 1], peaks[:, 0]]
log.info(
    "Peaks: %d found  (kernel=%d  min_threshold=%.4f)",
    len(peaks),
    PEAK_KERNEL_SIZE,
    MIN_PEAK_THRESHOLD,
)

if len(peaks):
    log.info(
        "Peak patch-grid (x,y): %s",
        list(zip(peaks[:, 0].tolist(), peaks[:, 1].tolist())),
    )

    log.info(
        "Peak similarities: %s",
        peaks_sim.cpu().numpy().tolist(),
    )

# ── Figure ────────────────────────────────────────────────────────────────────
density_np = density_map.cpu().numpy()
pooled_np = pooled.cpu().numpy()
diff_np = (density_map - pooled).cpu().numpy()
is_peak_np = is_peak.cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

im0 = axes[0].imshow(density_np, cmap="jet", aspect="auto")
axes[0].set_title("Density map (input to NMS)", fontsize=10)
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], shrink=0.75, pad=0.02)

im1 = axes[1].imshow(pooled_np, cmap="jet", aspect="auto")
axes[1].set_title(f"Max-pooled map (kernel {PEAK_KERNEL_SIZE}×{PEAK_KERNEL_SIZE})", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], shrink=0.75, pad=0.02)

im2 = axes[2].imshow(diff_np, cmap="coolwarm", aspect="auto")
axes[2].set_title("density − pooled\n(= 0 at local maxima)", fontsize=10)
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], shrink=0.75, pad=0.02)

axes[3].imshow(is_peak_np.astype(np.float32), cmap="Greys_r", aspect="auto")
if len(peaks):
    axes[3].scatter(
        peaks[:, 0].cpu().numpy(),
        peaks[:, 1].cpu().numpy(),
        c="red",
        s=120,
        marker="x",
        linewidths=2,
        zorder=5,
    )
axes[3].set_title(f"Peak mask  ({len(peaks)} peaks)", fontsize=10)
axes[3].axis("off")

plt.suptitle(
    f"Step 4: Max-pool NMS  |  kernel={PEAK_KERNEL_SIZE}  min_threshold={MIN_PEAK_THRESHOLD}",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Step 5 — Final Detection Result

In [ ]:
# ── Three-panel detection figure ──────────────────────────────────────────────
scale = [i / IMG_SIZE for i in query_imgs[0].size]
peaks_np = peaks.cpu().numpy()
px_x_vis = (peaks_np[:, 0] + 0.5) * PATCH_SIZE * scale[0] if len(peaks_np) else []
px_y_vis = (peaks_np[:, 1] + 0.5) * PATCH_SIZE * scale[1] if len(peaks_np) else []

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    f"Instance Detection  |  DINOv{DINO_VERSION[1]}-{DINO_SIZE}  block {LAYER_IDX}  "
    f"mode={EXEMPLAR_MODE}  threshold={DENSITY_THRESHOLD}  masked={EXEMPLAR_MASK_PATH is not None}",
    fontsize=10,
)

# Exemplar with mask
axes[0].imshow(exemplar_img)
if pixel_mask is not None:
    axes[0].imshow(
        np.array(
            Image.fromarray((pixel_mask.astype(np.uint8) * 255)).resize(
                exemplar_img.size, Image.NEAREST
            )
        ),
        alpha=0,  # invisible; just shows structure
    )
    # Draw mask contour via imshow with alpha
    mask_big = (
        np.array(
            Image.fromarray(pixel_mask.astype(np.uint8) * 255).resize(
                exemplar_img.size, Image.NEAREST
            )
        )
        > 0
    )
    g_ov = np.zeros((*mask_big.shape, 4), dtype=np.float32)
    g_ov[mask_big] = [0.2, 0.9, 0.2, 0.4]
    axes[0].imshow(g_ov)
axes[0].set_title("Exemplar  (green = masked region used)", fontsize=10)
axes[0].axis("off")

im = axes[1].imshow(density_np, cmap="jet", interpolation="nearest")
axes[1].set_title("Density map (patch grid)", fontsize=10)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(query_imgs[0])
if len(peaks_np):
    axes[2].scatter(
        px_x_vis,
        px_y_vis,
        c="red",
        s=120,
        marker="o",
        linewidths=1.5,
        edgecolors="white",
        zorder=5,
    )
axes[2].set_title(f"Detections — {len(peaks_np)} found", fontsize=10)
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Parameter Sweep 1 — Mask Patch Threshold

The `MASK_PATCH_THRESHOLD` controls how much of a patch's pixels must be True for the
patch to be included in the exemplar.  Low values include borderline patches; high values
restrict to the object core.  We sweep this and show how the number of masked patches
and the resulting density map change.

In [ ]:
# ── Mask threshold sweep ───────────────────────────────────────────────────────
MASK_THRESHOLDS = [0.05, 0.15, 0.30, 0.50, 0.75, 0.95]

fig, axes = plt.subplots(3, len(MASK_THRESHOLDS), figsize=(len(MASK_THRESHOLDS) * 4, 13))

for col, mthr in enumerate(MASK_THRESHOLDS):
    if pixel_mask is not None:
        pm = pixel_mask_to_patch_mask(pixel_mask, ex_h, ex_w, IMG_SIZE, threshold=mthr)
        pm_flat = torch.from_numpy(pm.reshape(-1)).to(exemplar_tokens.device)
        ex_tok_m = exemplar_tokens[pm_flat] if pm_flat.any() else exemplar_tokens
    else:
        pm = np.ones((ex_h, ex_w), dtype=bool)
        ex_tok_m = exemplar_tokens

    feat_m = compute_exemplar_features(ex_tok_m, mode=EXEMPLAR_MODE, k=EXEMPLAR_K)
    dm_m = compute_density_map(query_tokens, feat_m, q_h, q_w, DENSITY_THRESHOLD)
    pk_m = extract_peaks(dm_m, PEAK_KERNEL_SIZE, MIN_PEAK_THRESHOLD)
    dm_np_m = dm_m.cpu().numpy()

    # Row 0: patch mask
    axes[0, col].imshow(pm.astype(np.float32), cmap="Greens", vmin=0, vmax=1, aspect="auto")
    axes[0, col].set_title(f"mask_thr={mthr}\n{pm.sum()}/{ex_h * ex_w} patches", fontsize=9)
    axes[0, col].axis("off")

    # Row 1: density map
    im = axes[1, col].imshow(dm_np_m, cmap="jet", aspect="auto")
    axes[1, col].set_title(f"{(dm_np_m > 0).sum()} active patches", fontsize=9)
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], shrink=0.75, pad=0.02)

    # Row 2: detection overlay
    axes[2, col].imshow(display_q0)
    axes[2, col].imshow(heat_overlay(display_q0, upsample_map(dm_np_m, IMG_SIZE)))
    if len(pk_m):
        px_m = (pk_m[:, 0].float() + 0.5) * PATCH_SIZE
        py_m = (pk_m[:, 1].float() + 0.5) * PATCH_SIZE
        axes[2, col].scatter(
            px_m.cpu(),
            py_m.cpu(),
            c="red",
            s=80,
            marker="o",
            linewidths=1.5,
            edgecolors="white",
            zorder=5,
        )
    axes[2, col].set_title(f"{len(pk_m)} peak(s)", fontsize=9)
    axes[2, col].axis("off")

plt.suptitle(
    f"Sweep: mask_patch_threshold  |  mode={EXEMPLAR_MODE}  density_thr={DENSITY_THRESHOLD}",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Parameter Sweep 2 — Density Threshold

In [ ]:
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

fig, axes = plt.subplots(2, len(THRESHOLDS), figsize=(len(THRESHOLDS) * 4, 9))

for col, thr in enumerate(THRESHOLDS):
    dm_thr = compute_density_map(query_tokens, feat_masked, q_h, q_w, threshold=thr)
    pk_thr = extract_peaks(dm_thr, PEAK_KERNEL_SIZE, MIN_PEAK_THRESHOLD)
    dm_np = dm_thr.cpu().numpy()
    pk_np = pk_thr.cpu().numpy()

    im = axes[0, col].imshow(dm_np, cmap="jet", aspect="auto")
    axes[0, col].set_title(f"threshold={thr}\n{(dm_np > 0).sum()} active", fontsize=9)
    axes[0, col].axis("off")
    plt.colorbar(im, ax=axes[0, col], shrink=0.75, pad=0.02)

    axes[1, col].imshow(display_q0)
    axes[1, col].imshow(heat_overlay(display_q0, upsample_map(dm_np, IMG_SIZE)))
    if len(pk_np):
        axes[1, col].scatter(
            (pk_np[:, 0] + 0.5) * PATCH_SIZE,
            (pk_np[:, 1] + 0.5) * PATCH_SIZE,
            c="red",
            s=80,
            marker="o",
            linewidths=1.5,
            edgecolors="white",
            zorder=5,
        )
    axes[1, col].set_title(f"{len(pk_np)} peak(s)", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(
    f"Sweep: density_threshold  |  mode={EXEMPLAR_MODE}  masked exemplar",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Parameter Sweep 3 — Transformer Block (Layer Index)

In [ ]:
LAYER_INDICES = [6, 12, 16, 20, 22, 23]

fig, axes = plt.subplots(2, len(LAYER_INDICES), figsize=(len(LAYER_INDICES) * 4, 9))

for col, layer in enumerate(LAYER_INDICES):
    ex_tok_l, ex_h_l, ex_w_l = extract_patch_tokens(encoder, exemplar_img, layer)
    q_tok_l, q_h_l, q_w_l = extract_patch_tokens(encoder, query_imgs[0], layer)

    # Re-project mask to potentially different grid (same size here, kept for correctness)
    if pixel_mask is not None:
        pm_l = pixel_mask_to_patch_mask(pixel_mask, ex_h_l, ex_w_l, IMG_SIZE, MASK_PATCH_THRESHOLD)
        pm_flat_l = torch.from_numpy(pm_l.reshape(-1)).to(ex_tok_l.device)
        ex_tok_l_m = ex_tok_l[pm_flat_l] if pm_flat_l.any() else ex_tok_l
    else:
        ex_tok_l_m = ex_tok_l

    feat_l = compute_exemplar_features(ex_tok_l_m, mode=EXEMPLAR_MODE, k=EXEMPLAR_K)
    dm_l = compute_density_map(q_tok_l, feat_l, q_h_l, q_w_l, DENSITY_THRESHOLD)
    pk_l = extract_peaks(dm_l, PEAK_KERNEL_SIZE, MIN_PEAK_THRESHOLD)
    dm_l_np = dm_l.cpu().numpy()

    im = axes[0, col].imshow(dm_l_np, cmap="jet", aspect="auto")
    axes[0, col].set_title(f"block {layer}", fontsize=9)
    axes[0, col].axis("off")
    plt.colorbar(im, ax=axes[0, col], shrink=0.75, pad=0.02)

    axes[1, col].imshow(display_q0)
    axes[1, col].imshow(heat_overlay(display_q0, upsample_map(dm_l_np, IMG_SIZE)))
    if len(pk_l):
        axes[1, col].scatter(
            (pk_l[:, 0].float().cpu() + 0.5) * PATCH_SIZE,
            (pk_l[:, 1].float().cpu() + 0.5) * PATCH_SIZE,
            c="red",
            s=80,
            marker="o",
            linewidths=1.5,
            edgecolors="white",
            zorder=5,
        )
    axes[1, col].set_title(f"{len(pk_l)} peak(s)", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(
    f"Sweep: layer_idx  |  mode={EXEMPLAR_MODE}  masked exemplar  density_thr={DENSITY_THRESHOLD}",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Parameter Sweep 4 — Aggregation Mode (mean vs K-means)

In [ ]:
MODE_CONFIGS: list[tuple[str, int]] = [
    ("mean", 1),
    ("kmeans", 2),
    ("kmeans", 3),
    ("kmeans", 5),
]

fig, axes = plt.subplots(2, len(MODE_CONFIGS), figsize=(len(MODE_CONFIGS) * 5, 10))

for col, (mode, k) in enumerate(MODE_CONFIGS):
    feat_m = compute_exemplar_features(exemplar_tokens_masked, mode=mode, k=k)
    dm_m = compute_density_map(query_tokens, feat_m, q_h, q_w, DENSITY_THRESHOLD)
    pk_m = extract_peaks(dm_m, PEAK_KERNEL_SIZE, MIN_PEAK_THRESHOLD)
    dm_m_np = dm_m.cpu().numpy()

    label = "mean" if mode == "mean" else f"kmeans K={k}"

    im = axes[0, col].imshow(dm_m_np, cmap="jet", aspect="auto")
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].axis("off")
    plt.colorbar(im, ax=axes[0, col], shrink=0.75, pad=0.02)

    axes[1, col].imshow(display_q0)
    axes[1, col].imshow(heat_overlay(display_q0, upsample_map(dm_m_np, IMG_SIZE)))
    if len(pk_m):
        axes[1, col].scatter(
            (pk_m[:, 0].float().cpu() + 0.5) * PATCH_SIZE,
            (pk_m[:, 1].float().cpu() + 0.5) * PATCH_SIZE,
            c="red",
            s=80,
            marker="o",
            linewidths=1.5,
            edgecolors="white",
            zorder=5,
        )
    axes[1, col].set_title(f"{len(pk_m)} peak(s)", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(
    f"Sweep: aggregation mode  |  block={LAYER_IDX}  density_thr={DENSITY_THRESHOLD}  masked exemplar",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Multi-Query Detection

Run the full (masked) pipeline across all query images using the same exemplar.

In [ ]:
# ── Run on all queries ─────────────────────────────────────────────────────────
query_results: list[dict] = []
for path, q_img in zip(QUERY_PATHS, query_imgs):
    q_tok_f, q_h_f, q_w_f = extract_patch_tokens(encoder, q_img, LAYER_IDX)
    dm_f = compute_density_map(q_tok_f, feat_masked, q_h_f, q_w_f, DENSITY_THRESHOLD)
    pk_f = extract_peaks(dm_f, PEAK_KERNEL_SIZE, MIN_PEAK_THRESHOLD)
    query_results.append({"path": path, "img": q_img, "density": dm_f, "peaks": pk_f})
    log.info("%s → %d peak(s)  density max=%.3f", path.split("/")[-1], len(pk_f), dm_f.max().item())

nq = len(query_results)
fig, axes = plt.subplots(3, nq, figsize=(nq * 5, 15))
if nq == 1:
    axes = axes.reshape(3, 1)

for col, qr in enumerate(query_results):
    dm_np_f = qr["density"].cpu().numpy()
    pk_np_f = qr["peaks"].cpu().numpy()
    disp_qf = np.array(qr["img"].resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

    axes[0, col].imshow(qr["img"])
    axes[0, col].set_title(qr["path"].split("/")[-1], fontsize=8)
    axes[0, col].axis("off")

    im = axes[1, col].imshow(dm_np_f, cmap="jet", aspect="auto")
    axes[1, col].set_title(f"density max={dm_np_f.max():.3f}", fontsize=9)
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], shrink=0.75, pad=0.02)

    axes[2, col].imshow(qr["img"])
    if len(pk_np_f):
        axes[2, col].scatter(
            (pk_np_f[:, 0] + 0.5) * PATCH_SIZE,
            (pk_np_f[:, 1] + 0.5) * PATCH_SIZE,
            c="red",
            s=120,
            marker="o",
            linewidths=1.5,
            edgecolors="white",
            zorder=5,
        )
    axes[2, col].set_title(f"{len(pk_np_f)} peak(s)", fontsize=9)
    axes[2, col].axis("off")

plt.suptitle(
    f"Multi-query  |  DINOv{DINO_VERSION[1]}-{DINO_SIZE}  block {LAYER_IDX}  "
    f"mode={EXEMPLAR_MODE}  masked exemplar",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.show()

## Peak Self-Similarity Heatmap Drilldown

For each detected peak in query 1, compute the cosine similarity of that peak's
feature vector against every other query patch.  This shows what the model is
"matching" against — the instance-detection analogue of the keypoint heatmap.

In [ ]:
# ── Drilldown on query 1 ───────────────────────────────────────────────────────
if len(peaks) == 0:
    log.warning("No peaks in query 1 — lower DENSITY_THRESHOLD to see heatmaps.")
else:
    n_peaks = len(peaks)
    disp_q_d = np.array(query_imgs[0].resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

    fig, axes = plt.subplots(2, n_peaks, figsize=(n_peaks * 4, 8))
    if n_peaks == 1:
        axes = axes.reshape(2, 1)

    for col, peak in enumerate(peaks):
        px, py = peak[0].item(), peak[1].item()
        peak_feat = query_tokens[py * q_w + px]  # (D,)

        sim_self = (query_tokens @ peak_feat).reshape(q_h, q_w).cpu().numpy()
        heat_self = upsample_map(sim_self, IMG_SIZE)
        blended = np.clip(
            disp_q_d / 255.0 * 0.45 + plt.get_cmap("jet")(heat_self)[..., :3] * 0.55, 0, 1
        )

        im = axes[0, col].imshow(sim_self, cmap="jet", vmin=sim_self.min(), vmax=1.0, aspect="auto")
        axes[0, col].set_title(
            f"Peak ({px},{py}) self-sim\nscore={density_np[py, px]:.3f}", fontsize=9
        )
        axes[0, col].axis("off")
        plt.colorbar(im, ax=axes[0, col], shrink=0.75, pad=0.02)

        axes[1, col].imshow(blended)
        axes[1, col].scatter(
            [(px + 0.5) * PATCH_SIZE],
            [(py + 0.5) * PATCH_SIZE],
            c="red",
            s=120,
            marker="o",
            linewidths=1.5,
            edgecolors="white",
            zorder=5,
        )
        axes[1, col].set_title("Overlay (heat + query)", fontsize=9)
        axes[1, col].axis("off")

    plt.suptitle(
        f"Peak self-similarity heatmaps  |  query 1  |  block {LAYER_IDX}",
        fontsize=12,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

## Stage 2 — Two-Stage Pipeline: Object Crop → Clip Detection

Using the Stage 1 density map to locate and crop the object, then running a second
DINO pass on the crop to find the white clips.

**Pipeline:**
1. Stage 1 density map → bounding box (all patches where density > 0) → pixel-space crop
2. Crop the query image (with configurable padding)
3. Extract DINO tokens from the *cropped* image
4. Run the instance-detection pipeline with a **clip exemplar** on the crop
5. Project clip peaks back into original-image pixel coordinates

In [ ]:
# ── Stage 2 parameters ────────────────────────────────────────────────────────
# Exemplar image showing a single white clip, plus its instance mask.
CLIP_EXEMPLAR_PATH: str = str(data_dir / f"{part_type}_clip_exemplar.jpg")
CLIP_EXEMPLAR_MASK_PATH: str | None = str(
    data_dir / "annotations" / f"{part_type}_clip_mask.npy"
)

# Padding (pixels) added around the Stage 1 bounding box before cropping.
CROP_PADDING_PX: int = 20

# Stage 2 density / NMS settings (tune independently from Stage 1).
CLIP_DENSITY_THRESHOLD: float = 0.3
CLIP_PEAK_KERNEL_SIZE: int = 5
CLIP_MIN_PEAK_THRESHOLD: float = 0.3

In [ ]:
# ── Crop query image using Stage 1 density map ────────────────────────────────

def density_to_crop_box(
    dm: torch.Tensor,
    grid_h: int,
    grid_w: int,
    patch_size: int,
    orig_w: int,
    orig_h: int,
    padding_px: int = 0,
) -> tuple[int, int, int, int]:
    """Convert a density map to a padded pixel-space bounding box.

    All patches where density > 0 define the hot region; the box is expanded
    by ``padding_px`` on every side and clamped to the image boundary.

    Returns
    -------
    (x0, y0, x1, y1) in original-image pixel coordinates (PIL crop convention).
    """
    dm_np = dm.cpu().numpy()
    rows, cols = np.where(dm_np > 0)
    if len(rows) == 0:
        log.warning("Stage 1 density map is all zero — returning full image.")
        return 0, 0, orig_w, orig_h

    # Patch-grid bounding box (inclusive row/col, exclusive end)
    r0, r1 = int(rows.min()), int(rows.max()) + 1
    c0, c1 = int(cols.min()), int(cols.max()) + 1

    # Convert patch → pixel in IMG_SIZE space, then scale to original resolution
    scale_x = orig_w / (grid_w * patch_size)
    scale_y = orig_h / (grid_h * patch_size)

    x0 = max(0,      int(c0 * patch_size * scale_x) - padding_px)
    y0 = max(0,      int(r0 * patch_size * scale_y) - padding_px)
    x1 = min(orig_w, int(c1 * patch_size * scale_x) + padding_px)
    y1 = min(orig_h, int(r1 * patch_size * scale_y) + padding_px)
    return x0, y0, x1, y1


# Crop all query images based on their individual Stage 1 density maps
query_crops: list[dict] = []
for qr in query_results:
    qimg = qr["img"]
    orig_w, orig_h = qimg.size
    box = density_to_crop_box(
        qr["density"], q_h, q_w, PATCH_SIZE, orig_w, orig_h, padding_px=CROP_PADDING_PX
    )
    x0, y0, x1, y1 = box
    crop = qimg.crop(box)
    query_crops.append({"path": qr["path"], "img": qimg, "crop": crop, "box": box})
    log.info(
        "%s → bbox (%d,%d,%d,%d)  crop size %dx%d px",
        qr["path"].split("/")[-1], x0, y0, x1, y1, x1 - x0, y1 - y0,
    )

# ── Visualise crop boxes ───────────────────────────────────────────────────────
nq = len(query_crops)
fig, axes = plt.subplots(2, nq, figsize=(nq * 6, 10))
if nq == 1:
    axes = axes.reshape(2, 1)

for col, qc in enumerate(query_crops):
    x0, y0, x1, y1 = qc["box"]

    axes[0, col].imshow(qc["img"])
    rect = mpatches.Rectangle(
        (x0, y0), x1 - x0, y1 - y0,
        linewidth=2, edgecolor="lime", facecolor="none",
    )
    axes[0, col].add_patch(rect)
    axes[0, col].set_title(
        f"{qc['path'].split('/')[-1]}\nStage 1 crop box (pad={CROP_PADDING_PX}px)", fontsize=9
    )
    axes[0, col].axis("off")

    axes[1, col].imshow(qc["crop"])
    axes[1, col].set_title(f"Cropped object  ({x1-x0}×{y1-y0} px)", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle("Stage 2 Prep: Object crop from Stage 1 density map", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Stage 2: load clip exemplar + build descriptor ────────────────────────────
clip_exemplar_img = Image.open(CLIP_EXEMPLAR_PATH).convert("RGB")
log.info("Clip exemplar: %s  (%dx%d px)", CLIP_EXEMPLAR_PATH, *clip_exemplar_img.size)

# Load clip mask (same format support as Stage 1)
clip_pixel_mask: np.ndarray | None = None
if CLIP_EXEMPLAR_MASK_PATH is not None:
    if CLIP_EXEMPLAR_MASK_PATH.endswith(".npy"):
        clip_raw_seg = np.load(CLIP_EXEMPLAR_MASK_PATH)   # (N, H, W) bool
        clip_raw_seg = clip_raw_seg.transpose(1, 2, 0)     # → (H, W, N)
    elif CLIP_EXEMPLAR_MASK_PATH.endswith(".npz"):
        clip_raw_seg = np.load(CLIP_EXEMPLAR_MASK_PATH)["segmaps"]
    else:
        clip_raw_seg = np.array(Image.open(CLIP_EXEMPLAR_MASK_PATH).convert("L"))[:, :, None] > 0
    clip_pixel_mask = clip_raw_seg.any(axis=2)
    log.info(
        "Clip mask: shape=%s  coverage=%.1f%%",
        clip_raw_seg.shape, 100.0 * clip_pixel_mask.mean(),
    )

# Extract patch tokens from the clip exemplar
clip_ex_tokens, clip_ex_h, clip_ex_w = extract_patch_tokens(encoder, clip_exemplar_img, LAYER_IDX)
log.info(
    "Clip exemplar tokens: %s  (grid %dx%d)", clip_ex_tokens.shape, clip_ex_h, clip_ex_w
)

# Project clip pixel mask → patch mask
if clip_pixel_mask is not None:
    clip_patch_mask = pixel_mask_to_patch_mask(
        clip_pixel_mask, clip_ex_h, clip_ex_w, IMG_SIZE, threshold=MASK_PATCH_THRESHOLD
    )
    clip_patch_mask_flat = torch.from_numpy(clip_patch_mask.reshape(-1)).to(clip_ex_tokens.device)
    clip_ex_tokens_masked = clip_ex_tokens[clip_patch_mask_flat]
    log.info(
        "Clip patch mask: %d / %d patches inside mask  (threshold=%.2f)",
        clip_patch_mask_flat.sum().item(), clip_ex_h * clip_ex_w, MASK_PATCH_THRESHOLD,
    )
else:
    clip_ex_tokens_masked = clip_ex_tokens
    log.info("No clip mask — using all %d clip exemplar patches.", clip_ex_h * clip_ex_w)

# Build clip descriptor
feat_clips = compute_exemplar_features(clip_ex_tokens_masked, mode=EXEMPLAR_MODE, k=EXEMPLAR_K)
log.info("Clip descriptor shape: %s", feat_clips.shape)

# ── Run Stage 2 on all cropped queries ────────────────────────────────────────
clip_results: list[dict] = []
for qc in query_crops:
    crop = qc["crop"]
    c_tok, c_h, c_w = extract_patch_tokens(encoder, crop, LAYER_IDX)
    dm_c = compute_density_map(c_tok, feat_clips, c_h, c_w, CLIP_DENSITY_THRESHOLD)
    pk_c = extract_peaks(dm_c, CLIP_PEAK_KERNEL_SIZE, CLIP_MIN_PEAK_THRESHOLD)

    # Scale factor from patch grid → crop pixels
    crop_w, crop_h = crop.size
    sx = crop_w / (c_w * PATCH_SIZE)
    sy = crop_h / (c_h * PATCH_SIZE)

    # Clip peak coordinates in crop space
    pk_np = pk_c.cpu().numpy()
    px_crop = (pk_np[:, 0] + 0.5) * PATCH_SIZE * sx if len(pk_np) else np.array([])
    py_crop = (pk_np[:, 1] + 0.5) * PATCH_SIZE * sy if len(pk_np) else np.array([])

    # Project back to original-image space
    x0, y0, _, _ = qc["box"]
    px_orig = px_crop + x0
    py_orig = py_crop + y0

    clip_results.append({
        "path": qc["path"],
        "img": qc["img"],
        "crop": crop,
        "box": qc["box"],
        "density": dm_c,
        "peaks_np": pk_np,
        "px_crop": px_crop,
        "py_crop": py_crop,
        "px_orig": px_orig,
        "py_orig": py_orig,
        "grid": (c_h, c_w),
    })
    log.info(
        "%s → %d clip(s) found  density_max=%.3f",
        qc["path"].split("/")[-1], len(pk_np), dm_c.max().item(),
    )

In [ ]:
# ── Stage 2 results visualisation ─────────────────────────────────────────────
# Per query: 4 panels — clip exemplar | density on crop | detections on crop | full image
nq = len(clip_results)
fig, axes = plt.subplots(nq, 4, figsize=(24, nq * 6))
if nq == 1:
    axes = axes.reshape(1, 4)

# Clip exemplar panel (constant, shown in col 0)
clip_display = np.array(clip_exemplar_img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

for row, cr in enumerate(clip_results):
    dm_np_c = cr["density"].cpu().numpy()
    c_h, c_w = cr["grid"]
    crop_display = np.array(cr["crop"].resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

    # Col 0: clip exemplar with mask
    axes[row, 0].imshow(clip_exemplar_img)
    if clip_pixel_mask is not None:
        mask_big_c = np.array(
            Image.fromarray(clip_pixel_mask.astype(np.uint8) * 255)
            .resize(clip_exemplar_img.size, Image.NEAREST)
        ) > 0
        g_ov_c = np.zeros((*mask_big_c.shape, 4), dtype=np.float32)
        g_ov_c[mask_big_c] = [0.2, 0.9, 0.2, 0.4]
        axes[row, 0].imshow(g_ov_c)
    axes[row, 0].set_title("Clip exemplar (green = masked region)", fontsize=9)
    axes[row, 0].axis("off")

    # Col 1: clip density overlay on crop
    axes[row, 1].imshow(crop_display)
    axes[row, 1].imshow(heat_overlay(crop_display, upsample_map(dm_np_c, IMG_SIZE)))
    axes[row, 1].set_title(
        f"Stage 2 density on crop\nmax={dm_np_c.max():.3f}", fontsize=9
    )
    axes[row, 1].axis("off")

    # Col 2: clip detections on cropped image (native resolution)
    axes[row, 2].imshow(cr["crop"])
    if len(cr["peaks_np"]):
        axes[row, 2].scatter(
            cr["px_crop"], cr["py_crop"],
            c="red", s=120, marker="o", linewidths=1.5, edgecolors="white", zorder=5,
        )
    axes[row, 2].set_title(
        f"Clip detections on crop — {len(cr['peaks_np'])} found", fontsize=9
    )
    axes[row, 2].axis("off")

    # Col 3: full query with stage 1 bbox + stage 2 clip peaks projected back
    x0, y0, x1, y1 = cr["box"]
    axes[row, 3].imshow(cr["img"])
    rect = mpatches.Rectangle(
        (x0, y0), x1 - x0, y1 - y0,
        linewidth=2, edgecolor="lime", facecolor="none", label="Object (stage 1)",
    )
    axes[row, 3].add_patch(rect)
    if len(cr["peaks_np"]):
        axes[row, 3].scatter(
            cr["px_orig"], cr["py_orig"],
            c="red", s=120, marker="o", linewidths=1.5, edgecolors="white", zorder=5,
            label=f"Clips ({len(cr['peaks_np'])})",
        )
    axes[row, 3].legend(fontsize=8, loc="upper right")
    axes[row, 3].set_title(
        f"{cr['path'].split('/')[-1]}\nStage 1 bbox + stage 2 clip peaks", fontsize=9
    )
    axes[row, 3].axis("off")

plt.suptitle(
    f"Stage 2: Clip Detection on Cropped Object  |  "
    f"DINOv{DINO_VERSION[1]}-{DINO_SIZE}  block {LAYER_IDX}  mode={EXEMPLAR_MODE}",
    fontsize=12,
)
plt.tight_layout()
plt.show()